# Task 5 — AI Yoga Instructor (IMU classification)

Decide whether one yoga repetition was performed correctly, from wearable sensor data
sampled at 200 Hz: `ax ay az` (acceleration) and `wx wy wz` (angular velocity).

Statement: [`../qualification/task5_Can_You_Become_AI_Yoga_Instructor.md`](../qualification/task5_Can_You_Become_AI_Yoga_Instructor.md)
Reasoning behind every choice here: [`task5_ai_yoga_instructor.md`](./task5_ai_yoga_instructor.md)

**Before running:** put `X_train.csv`, `y_train.csv` and `X_test.csv` next to this notebook.
Dataset links are in the statement (Google Drive).

In [ ]:
X_TRAIN = "X_train.csv"
Y_TRAIN = "y_train.csv"
X_TEST  = "X_test.csv"
OUT     = "solution.csv"    # columns: id, label

RANDOM_STATE = 42

In [ ]:
import numpy as np
import pandas as pd

Xtr_raw = pd.read_csv(X_TRAIN)
ytr_raw = pd.read_csv(Y_TRAIN)
Xte_raw = pd.read_csv(X_TEST)

print("X_train", Xtr_raw.shape, "| y_train", ytr_raw.shape, "| X_test", Xte_raw.shape)
print(Xtr_raw.columns.tolist())

# THE FIRST COMMAND OF THE ROUND.
# The statement describes a 200 Hz time series per id, but its example shows one row per id.
# This settles it and decides the whole pipeline.
rows_per_id = Xtr_raw.groupby("id").size()
print("rows per id -> min", rows_per_id.min(), "max", rows_per_id.max(), "median", rows_per_id.median())
IS_TIMESERIES = rows_per_id.max() > 1
print("time series per id:", IS_TIMESERIES)

## Features

If each `id` holds many rows, the label belongs to the repetition, not the row, so every
repetition must collapse to a single feature vector.

Spread matters more than level here: "correct vs incorrect pose" is about how the motion is
shaped, not its average value. Magnitudes are added because they survive the sensor being
mounted at a different angle.

In [ ]:
SIG = ["ax", "ay", "az", "wx", "wy", "wz"]

def featurise(df: pd.DataFrame) -> pd.DataFrame:
    if not IS_TIMESERIES:
        return df.set_index("id").sort_index()          # already one row per repetition

    d = df.copy()
    # Rotation-robust: total magnitude does not care how the sensor was oriented.
    d["amag"] = np.sqrt(d.ax**2 + d.ay**2 + d.az**2)
    d["wmag"] = np.sqrt(d.wx**2 + d.wy**2 + d.wz**2)
    cols = SIG + ["amag", "wmag"]

    g = d.groupby("id")[cols]
    agg = g.agg(["mean", "std", "min", "max", "median"])
    agg.columns = ["_".join(c) for c in agg.columns]     # flatten MultiIndex immediately

    agg["n_rows"] = d.groupby("id").size()               # duration, at a fixed 200 Hz
    for c in cols:
        agg[f"{c}_rms"]   = g[c].apply(lambda s: np.sqrt((s**2).mean()))
        agg[f"{c}_range"] = g[c].max() - g[c].min()

    # Coordination between axes: sloppy movement decorrelates.
    for a, b in [("ax","ay"), ("ax","az"), ("ay","az"), ("wx","wy"), ("wx","wz"), ("wy","wz")]:
        agg[f"corr_{a}_{b}"] = d.groupby("id").apply(lambda s: s[a].corr(s[b]))

    return agg.fillna(0.0).sort_index()                  # std of a 1-row group is NaN

Xtr = featurise(Xtr_raw)
Xte = featurise(Xte_raw)
ytr = ytr_raw.set_index("id").loc[Xtr.index, "label"]

print("features:", Xtr.shape, "| test:", Xte.shape)
print("label balance:", ytr.value_counts().to_dict())

## Validation

If one repetition spans many rows, its rows must never sit on both sides of a split — they are
5 ms apart and nearly identical, so a random split scores itself on near-duplicates and returns
a fake number close to 0.99.

Here the features are already one row per `id`, so plain stratified k-fold is correct. The
grouped splitter is shown for the case where you model raw rows instead.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

candidates = {
    "hgb_lr06": HistGradientBoostingClassifier(max_iter=400, learning_rate=0.06,
                                               random_state=RANDOM_STATE),
    "hgb_lr03": HistGradientBoostingClassifier(max_iter=800, learning_rate=0.03,
                                               random_state=RANDOM_STATE),
}

best, best_score = None, -1.0
for name, clf in candidates.items():
    s = cross_val_score(clf, Xtr, ytr, cv=cv, scoring="accuracy", n_jobs=-1)
    se = s.std(ddof=1) / np.sqrt(len(s))
    print(f"{name}: accuracy {s.mean():.4f} +/- {se:.4f}")
    if s.mean() > best_score:
        best, best_score = clf, s.mean()

print("chosen:", best)

In [ ]:
best.fit(Xtr, ytr)

# Which features carry the signal, by permutation on a held-out split.
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split

Xa, Xb, ya, yb = train_test_split(Xtr, ytr, test_size=0.25,
                                  stratify=ytr, random_state=RANDOM_STATE)
probe = HistGradientBoostingClassifier(max_iter=300, random_state=RANDOM_STATE).fit(Xa, ya)
imp = permutation_importance(probe, Xb, yb, n_repeats=5, random_state=RANDOM_STATE)
top = pd.Series(imp.importances_mean, index=Xtr.columns).sort_values(ascending=False)
print(top.head(12))

## Write the submission\n\n`id,label` in ascending id order, labels `0`/`1`, no index column.

In [ ]:
pred = best.predict(Xte)

sub = pd.DataFrame({"id": Xte.index, "label": pred.astype(int)}).sort_values("id")
sub.to_csv(OUT, index=False)

assert list(sub.columns) == ["id", "label"]
assert len(sub) == Xte_raw.id.nunique(), f"{len(sub)} rows, expected {Xte_raw.id.nunique()}"
assert set(sub.label) <= {0, 1}
assert sub.id.tolist() == sorted(sub.id.tolist())
print(f"{OUT}: {len(sub)} rows OK")
print(sub.label.value_counts().to_dict())
sub.head()